In [1]:
%pip install torch transformers datasets pandas numpy
%pip install datasets transformers torch

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/204.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/204.1 MB 3.4 MB/s eta 0:01:01
   ---------------------------------------- 1.0/204.1 MB 3.1 MB/s eta 0:01:05
   ---------------------------------------- 1.8/204.1 MB 3.1 MB/s eta 0:01:05
   ---------------------------------------- 2.4/204.1 MB 3.1 MB/s eta 0:01:05
    --------------------------------------- 3.1/204.1 MB 3.1 MB/s eta 0:01:05
    --------------------------------------- 3.7/204.1 MB 3.1 MB/s eta 0:01:05
    --------------------------------------- 4.5/204.1 MB 3.1 MB/s eta 0:01:04
    --------------------------------------- 5.0/204.1 MB 3.1 MB/s eta 0:01:04
   - -------------------------------------- 5.2/204.1 MB 3.1 MB/s eta 0:01:04
   - -------------------------------------- 6.0/204.1 MB 3.0 MB/s eta 0:01:07
   - -------------------------------------- 6.3/204.1 MB 2.8 MB/s eta 0


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
import torch

C:\Users\sarah\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = load_dataset("rajpurkar/squad")


C:\Users\sarah\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sarah\.cache\huggingface\hub\datasets--rajpurkar--squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating validation split: 100%|██████████| 10570/10570 [00:00<00:00, 783379.45 examples/s]


In [ ]:

import numpy as np
def load_glove_embeddings(file_path):
    embeddings_index = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embeddings_index[word] = coefs
    return embeddings_index


glove_file_path = 'path/to/glove.6B.100d.txt'


glove_embeddings = load_glove_embeddings(glove_file_path)
print(f"Loaded {len(glove_embeddings)} word vectors.")

In [ ]:

embedding_matrix = np.zeros((len(tokenizer.vocab), 100))  # 100 because we're using GloVe 100D embeddings

for word, index in tokenizer.vocab.items():
    # If the word is in GloVe embeddings, assign it
    if word in glove_embeddings:
        embedding_matrix[index] = glove_embeddings[word]
    else:
        # If the word isn't in GloVe, leave it as a zero vector (or random initialization)
        embedding_matrix[index] = np.random.normal(scale=0.6, size=(100,))

print(f"Embedding matrix shape: {embedding_matrix.shape}")

In [4]:
# Check the structure of the dataset
print(dataset)

# Access the train and validation sets
train_dataset = dataset['train']
validation_dataset = dataset['validation']

# Inspect a sample from the train set
print(train_dataset[0])


DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})
{'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome

In [5]:
# import pandas as pd

# # Convert train dataset to DataFrame
# train_df = pd.DataFrame(dataset['train'])

# # Sample a subset of data (e.g., 5k rows)
# train_df = train_df.sample(n=5000, random_state=42)

# # Extract question and answer columns in a cleaner format
# questions = train_df['question'].tolist()
# answers = [ans['text'][0] for ans in train_df['answers']]  # Extract answer text from the 'answers' field

# # Create a new DataFrame with question-answer pairs
# qa_df = pd.DataFrame({
#     'question': questions,
#     'answer': answers
# })

# # Display the first few rows to see the cleaned format
# print(qa_df.head())

# Extract questions and answers in a cleaner format
questions = [entry['question'] for entry in train_dataset]
answers = [entry['answers']['text'][0] for entry in train_dataset]  # Using the first answer if there are multiple answers

# Create a new DataFrame with question-answer pairs
import pandas as pd
qa_df = pd.DataFrame({
    'question': questions,
    'answer': answers
})

# Display the first few rows to see the cleaned format
print(qa_df.head())




                                            question  \
0  To whom did the Virgin Mary allegedly appear i...   
1  What is in front of the Notre Dame Main Building?   
2  The Basilica of the Sacred heart at Notre Dame...   
3                  What is the Grotto at Notre Dame?   
4  What sits on top of the Main Building at Notre...   

                                    answer  
0               Saint Bernadette Soubirous  
1                a copper statue of Christ  
2                        the Main Building  
3  a Marian place of prayer and reflection  
4       a golden statue of the Virgin Mary  


In [6]:
# Load the pre-trained BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenize the questions and answers
qa_df['question_tokens'] = qa_df['question'].apply(lambda x: tokenizer.encode(x, truncation=True, padding='max_length', max_length=32))
qa_df['answer_tokens'] = qa_df['answer'].apply(lambda x: tokenizer.encode(x, truncation=True, padding='max_length', max_length=32))

# Check the first few tokenized questions and answers
print(qa_df[['question_tokens', 'answer_tokens']].head())


C:\Users\sarah\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sarah\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


                                     question_tokens  \
0  [101, 2000, 3183, 2106, 1996, 6261, 2984, 9382...   
1  [101, 2054, 2003, 1999, 2392, 1997, 1996, 1028...   
2  [101, 1996, 13546, 1997, 1996, 6730, 2540, 201...   
3  [101, 2054, 2003, 1996, 24665, 23052, 2012, 10...   
4  [101, 2054, 7719, 2006, 2327, 1997, 1996, 2364...   

                                       answer_tokens  
0  [101, 3002, 16595, 9648, 4674, 2061, 12083, 97...  
1  [101, 1037, 6967, 6231, 1997, 4828, 102, 0, 0,...  
2  [101, 1996, 2364, 2311, 102, 0, 0, 0, 0, 0, 0,...  
3  [101, 1037, 14042, 2173, 1997, 7083, 1998, 918...  
4  [101, 1037, 3585, 6231, 1997, 1996, 6261, 2984...  


In [7]:
# Convert tokenized questions and answers into PyTorch tensors
questions_tensor = torch.tensor(qa_df['question_tokens'].tolist())
answers_tensor = torch.tensor(qa_df['answer_tokens'].tolist())

In [8]:
from torch.utils.data import DataLoader, TensorDataset

# Create a TensorDataset
dataset = TensorDataset(questions_tensor, answers_tensor)

# Create a DataLoader for batching
batch_size = 8
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Check the first batch to ensure everything is correct
for batch in data_loader:
    print(batch)
    break


[tensor([[  101,  1999,  2054,  2301,  2106,  1996,  7788,  2806, 13236,  2000,
          2022,  2109,  1029,   102,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0],
        [  101,  2862,  2048, 15955,  2000, 16933, 29096,  2099,  9693,  1999,
         15041,  5097,  1012,   102,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0],
        [  101,  2129,  2024, 19765,  4340,  2747,  1029,   102,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0],
        [  101,  2043,  2001,  1996, 10864,  2231,  1997,  1996,  3537,  2976,
          8936,  9240,  1029,   102,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0, 

In [16]:
# import torch
# import torch.nn as nn
# import torch.optim as optim

# class ImprovedQuestionAnsweringModel(nn.Module):
#     def __init__(self, vocab_size, embed_size, hidden_size, num_layers=2):  # Fixed typo in method name
#         super(ImprovedQuestionAnsweringModel, self).__init__()

#         self.embedding = nn.Embedding(vocab_size, embed_size)

#         self.lstm = nn.LSTM(embed_size, hidden_size, num_layers=num_layers, 
#                             batch_first=True, bidirectional=True)

#         self.fc = nn.Linear(hidden_size * 2, vocab_size)  # *2 because of bidirectionality

#     def forward(self, x):
#         x = self.embedding(x)  # Embed input token IDs into vectors
#         lstm_out, _ = self.lstm(x)  # Pass through LSTM layers
#         out = self.fc(lstm_out)  # Apply the fully connected layer
#         return out


# vocab_size = len(tokenizer.vocab)  # Size of the tokenizer's vocabulary
# embed_size = 128  # Size of the word embeddings
# hidden_size = 128  # Hidden layer size (this can be adjusted for better performance)
# num_layers = 2  # Number of LSTM layers (stacked)


# model = ImprovedQuestionAnsweringModel(vocab_size, embed_size, hidden_size, num_layers)


# device = torch.device("cpu")
# model.to(device)

# loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)  # Ignore padding tokens
# optimizer = optim.Adam(model.parameters(), lr=0.001)



import torch
import torch.nn as nn
import torch.optim as optim


class ImprovedQuestionAnsweringModelWithGloVe(nn.Module):
    def init(self, vocab_size, embedding_matrix, hidden_size, num_layers=2):
        super(ImprovedQuestionAnsweringModelWithGloVe, self).init()


        self.embedding = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix, dtype=torch.float32), freeze=False)


        self.lstm = nn.LSTM(100, hidden_size, num_layers=num_layers, 
                            batch_first=True, bidirectional=True)

        self.fc = nn.Linear(hidden_size * 2, vocab_size)  # *2 because of bidirectionality

    def forward(self, x):
        x = self.embedding(x)  # Embed input token IDs into vectors
        lstmout,  = self.lstm(x)  # Pass through LSTM layers
        out = self.fc(lstm_out)  # Apply the fully connected layer
        return out

Hyperparameters
vocab_size = len(tokenizer.vocab)  # Size of the tokenizer's vocabulary
hidden_size = 256  # Hidden layer size
num_layers = 2     # Number of LSTM layers (stacked)


model = ImprovedQuestionAnsweringModelWithGloVe(vocab_size=len(tokenizer.vocab), 
                                                embedding_matrix=embedding_matrix, 
                                                hidden_size=hidden_size, 
                                                num_layers=num_layers)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)  # Ignore padding tokens
optimizer = optim.Adam(model.parameters(), lr=5e-5)



In [ ]:
train_model(model, data_loader, optimizer, loss_fn, epochs=5)

In [17]:
# # Training loop
# def train_model(model, data_loader, optimizer, loss_fn, epochs=5):
#     model.train()  # Set model to training mode
#     for epoch in range(epochs):
#         total_loss = 0
#         for questions_batch, answers_batch in data_loader:
#             optimizer.zero_grad()

#             # Forward pass
#             output = model(questions_batch)
#             output = output.view(-1, vocab_size)  # Flatten the output to (batch_size * seq_len, vocab_size)
#             answers_batch = answers_batch.view(-1)  # Flatten answers to match the output shape

#             # Calculate the loss
#             loss = loss_fn(output, answers_batch)
#             loss.backward()  # Backpropagation
#             optimizer.step()  # Update the weights

#             total_loss += loss.item()

#         print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss / len(data_loader)}")

# # Train the model
# train_model(model, data_loader, optimizer, loss_fn)


def train_model(model, data_loader, optimizer, loss_fn, epochs=5):
    model.train()  # Set model to training mode
    for epoch in range(epochs):
        total_loss = 0
        for questions_batch, answers_batch in data_loader:
            questions_batch = questions_batch.to(device)
            answers_batch = answers_batch.to(device)

            optimizer.zero_grad()

            # Forward pass
            output = model(questions_batch)
            output = output.view(-1, vocab_size)  # Flatten the output to (batch_size * seq_len, vocab_size)
            answers_batch = answers_batch.view(-1)  # Flatten answers to match the output shape

            # Calculate the loss
            loss = loss_fn(output, answers_batch)
            loss.backward()  # Backpropagation
            optimizer.step()  # Update the weights

            total_loss += loss.item()

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss / len(data_loader)}")


train_model(model, data_loader, optimizer, loss_fn)

Epoch [1/5], Loss: 5.592956181312805


KeyboardInterrupt: 

In [14]:
def evaluate_model(model, data_loader):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():  # No need to track gradients during evaluation
        for questions_batch, answers_batch in data_loader:
            # Forward pass
            output = model(questions_batch)
            output = output.argmax(dim=2)  # Get the predicted tokens with the highest probability
            output = output.view(-1).cpu().numpy()  # Flatten and move to CPU for comparison

            answers_batch = answers_batch.view(-1).cpu().numpy()  # Flatten answers for comparison

            correct += (output == answers_batch).sum()
            total += len(answers_batch)

    accuracy = correct / total
    print(f"Accuracy: {accuracy * 100:.2f}%")

# Evaluate the model
evaluate_model(model, data_loader)


Accuracy: 6.54%


In [11]:
def evaluate_model(model, data_loader, loss_fn, tokenizer, device):
    model.eval()  # Set the model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0
    exact_match = 0  # Counter for exact match

    with torch.no_grad():  # No need to track gradients during evaluation
        for questions_batch, answers_batch in data_loader:
            # Move data to GPU if available
            questions_batch = questions_batch.to(device)
            answers_batch = answers_batch.to(device)

            # Forward pass
            output = model(questions_batch)
            output = output.view(-1, len(tokenizer.vocab))  # Flatten to (batch_size * seq_len, vocab_size)
            answers_batch = answers_batch.view(-1)  # Flatten answers to match the output shape
            loss = loss_fn(output, answers_batch)
            total_loss += loss.item()

            # Get predicted tokens (argmax along the vocab dimension)
            predicted_answer = output.argmax(dim=1).cpu().numpy()
            answers_batch = answers_batch.cpu().numpy()

            # Calculate exact match (EM) and total correct predictions
            correct += (predicted_answer == answers_batch).sum()
            total += len(answers_batch)

            # Exact match calculation
            for pred, true in zip(predicted_answer, answers_batch):
                if pred == true:
                    exact_match += 1

    # Calculate final metrics
    accuracy = correct / total
    em_score = exact_match / total  # Exact Match ratio
    avg_loss = total_loss / len(data_loader)

    print(f"Evaluation Metrics:")
    print(f"  - Loss: {avg_loss:.4f}")
    print(f"  - Accuracy: {accuracy * 100:.2f}%")
    print(f"  - Exact Match (EM): {em_score * 100:.2f}%")

    return accuracy, em_score, avg_loss

In [13]:
device = torch.device("cpu")
model.to(device)


val_data_loader = DataLoader(
    TensorDataset(torch.tensor(validation_dataset['question_tokens'].tolist()), 
                  torch.tensor(validation_dataset['answer_tokens'].tolist())),
    batch_size=8, shuffle=False)


evaluate_model(model, val_data_loader, loss_fn, tokenizer, device)

KeyError: "Column question_tokens not in the dataset. Current columns in the dataset: ['id', 'title', 'context', 'question', 'answers']"